# 본문 CSV 쿼리별 통합 (Colab용)

월별로 저장된 `본문_bs4_*.csv` 파일을 쿼리별로 합쳐 `통합_본문_bs4_{query}.csv` 생성. 한글 파일명이 환경에 따라 다르게 저장될 수 있어 이름을 한 번 정리해서 처리함.

- 입력: `data/본문_bs4_{query}_{YYMMDD}_{YYMMDD}.csv`
- 출력: `data/통합_본문_bs4_{query}.csv`
- 보조 출력: `data/통합_본문_bs4_요약.csv`
- 특징: 쿼리별 월별 CSV 병합, 링크 기준 중복 제거, 날짜순 정렬


In [ ]:
# Colab에서 실행할 때만 아래 3줄 주석 해제 — 로컬/WSL에서는 그대로 두기
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os
import re
import unicodedata

import pandas as pd

# Colab Drive 폴더가 연결되어 있으면 그 경로 사용, 아니면 로컬 경로 사용
# 로컬/WSL에서는 except 쪽 경로 사용
try:
    PROJECT_DIR = Path('/content/drive/MyDrive/Text-data-Analysis_26-Spring')
    if not PROJECT_DIR.exists():
        raise FileNotFoundError
except Exception:
    PROJECT_DIR = Path('/home/carol/Text-data-Analysis_26-Spring')

os.chdir(PROJECT_DIR)  # 상대경로가 프로젝트 기준으로 잡히도록 작업 폴더 변경
print(f'현재 작업 폴더: {Path.cwd()}')

# 본문 수집 단계 결과물(본문_bs4_*.csv)이 모인 폴더 — 입력과 출력이 모두 이 안에 들어감
DATA_DIR = PROJECT_DIR / 'data' / 'crawling'
print(f'DATA_DIR: {DATA_DIR}')


In [ ]:
# Drive에서 한글 파일명이 다르게 인식될 수 있어, 비교 전에 같은 방식으로 정리함
def normalize_text(text):
    return unicodedata.normalize('NFC', str(text))


def normalize_name(path):
    return normalize_text(path.name)


# 파일명에서 쿼리/시작·끝 일자 분리
# 예: '본문_bs4_SKT_250401_250430.csv' -> query=SKT, start_ymd=250401, end_ymd=250430
def parse_body_filename(path):
    name = normalize_name(path)
    match = re.match(r'^본문_bs4_(.+)_(\d{6})_(\d{6})\.csv$', name)
    if not match:
        return None

    query, start_ymd, end_ymd = match.groups()
    return {
        'query': query,
        'period': f'{start_ymd}_{end_ymd}',
        'start_ym': f'20{start_ymd[:2]}.{start_ymd[2:4]}',
        'start_ymd': start_ymd,
        'end_ymd': end_ymd,
        'path': path,
    }


# data 폴더의 파일명들을 확인한 뒤 본문 CSV 파일만 골라냄
body_file_infos = []
for path in DATA_DIR.iterdir():
    if not path.is_file():
        continue
    info = parse_body_filename(path)
    if info:
        body_file_infos.append(info)

# 쿼리 → 시작 일자 순으로 정렬해 같은 쿼리의 월별 파일이 시간순으로 처리되도록 함
body_file_infos = sorted(body_file_infos, key=lambda x: (x['query'], x['start_ymd']))

if not body_file_infos:
    # 못 찾았을 때 DATA_DIR 안의 파일명을 일부 보여줘 파일명 문제를 확인할 수 있게 함
    sample_names = [normalize_name(p) for p in list(DATA_DIR.iterdir())[:20]]
    raise ValueError(f'본문_bs4_*.csv 파일을 찾지 못했습니다. DATA_DIR 확인 필요: {DATA_DIR}\n샘플 파일명: {sample_names}')

print(f'통합 대상 월별 본문 CSV: {len(body_file_infos)}개')
for info in body_file_infos:
    print(f"{info['query']} {info['period']} -> {info['path'].name}")


In [ ]:
# 쿼리별 파일 목록 확인 — 어느 월이 빠져있는지 눈으로 점검할 때 사용
file_index_df = pd.DataFrame([
    {
        'query': info['query'],
        'period': info['period'],
        'start_ym': info['start_ym'],
        'file_name': info['path'].name,
    }
    for info in body_file_infos
])

file_index_df


In [ ]:
# 쿼리별로 월별 CSV를 합쳐 하나의 통합 CSV로 저장
# 각 행에 어느 파일에서 왔는지 추적할 수 있도록 source_query/source_period/source_start_ym 컬럼 추가
summary_rows = []
merged_paths = []

for query, group_df in file_index_df.groupby('query', sort=True):
    query_infos = [info for info in body_file_infos if info['query'] == query]
    monthly_dfs = []

    print()
    print(f'=== {query} 통합 시작: {len(query_infos)}개 파일 ===')

    for info in query_infos:
        df = pd.read_csv(info['path'], encoding='utf-8-sig')
        # 통합 후에도 어느 파일에서 온 행인지 알 수 있도록 출처 컬럼 추가
        df['source_query'] = info['query']
        df['source_period'] = info['period']
        df['source_start_ym'] = info['start_ym']
        monthly_dfs.append(df)
        print(f"{info['period']}: {len(df)}행")

    merged_df = pd.concat(monthly_dfs, ignore_index=True)
    before_rows = len(merged_df)

    # 같은 쿼리 안에서 동일 링크가 여러 월에 잡히는 경우가 있어 link 기준 중복 제거
    # keep='first'로 가장 이른 월의 행을 살림
    if 'link' in merged_df.columns:
        merged_df = merged_df.drop_duplicates(subset=['link'], keep='first').reset_index(drop=True)

    duplicate_removed = before_rows - len(merged_df)

    # pubdate를 날짜 형식으로 바꿔 날짜순으로 정렬함. 날짜로 바꾸지 못한 값은 비워 둠
    if 'pubdate' in merged_df.columns:
        merged_df['pubdate'] = pd.to_datetime(merged_df['pubdate'], errors='coerce')
        merged_df = merged_df.sort_values('pubdate').reset_index(drop=True)

    save_path = DATA_DIR / f'통합_본문_bs4_{query}.csv'
    merged_df.to_csv(save_path, index=False, encoding='utf-8-sig')
    merged_paths.append(save_path)

    # 쿼리별 통합 결과 요약 (행수/중복 제거 건수/기간 범위)
    summary_rows.append({
        'query': query,
        'file_count': len(query_infos),
        'before_rows': before_rows,
        'merged_rows': len(merged_df),
        'duplicate_removed': duplicate_removed,
        'start_ym': min(info['start_ym'] for info in query_infos),
        'end_ym': max(info['start_ym'] for info in query_infos),
        'save_path': str(save_path),
    })

    print(f'저장 완료: {save_path}')
    print(f'행 수: {before_rows} -> {len(merged_df)} / 중복 제거 {duplicate_removed}건')

summary_df = pd.DataFrame(summary_rows).sort_values('query').reset_index(drop=True)
summary_df


In [ ]:
# 통합 결과 요약 저장 — 쿼리별 통합본의 메타데이터(행수, 중복 제거 건수, 기간)를 한 파일에 모아둠
summary_path = DATA_DIR / '통합_본문_bs4_요약.csv'
summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')

print(f'요약 저장 완료: {summary_path}')
print('생성된 통합 CSV:')
for path in merged_paths:
    print(path)


In [ ]:
# 전체 쿼리 통합본이 필요할 때만 사용
# 쿼리별 통합 CSV를 다시 합쳐 전체 분석용 파일 생성
# 단, SKT/SK텔레콤, LG U+/LG유플러스처럼 같은 대상을 두 쿼리로 수집한 경우 link 기준 중복 제거로
# 한쪽만 살아남으므로 쿼리별 분포 분석에는 부적합 — 분석 목적에 맞으면 True로 변경
MAKE_ALL_COMBINED = False

if MAKE_ALL_COMBINED:
    all_df = pd.concat([pd.read_csv(path, encoding='utf-8-sig') for path in merged_paths], ignore_index=True)
    # 쿼리 간에도 동일 링크가 중복될 수 있으므로 한 번 더 제거
    if 'link' in all_df.columns:
        all_df = all_df.drop_duplicates(subset=['link'], keep='first').reset_index(drop=True)
    if 'pubdate' in all_df.columns:
        all_df['pubdate'] = pd.to_datetime(all_df['pubdate'], errors='coerce')
        all_df = all_df.sort_values('pubdate').reset_index(drop=True)

    all_save_path = DATA_DIR / '통합_본문_bs4_전체.csv'
    all_df.to_csv(all_save_path, index=False, encoding='utf-8-sig')
    print(f'전체 통합본 저장 완료: {all_save_path}')
    print(f'전체 행 수: {len(all_df)}')
